# 08 · Other targets: Metal and Jetson

The same model also ships as two other builds. Neither needs the target device to *lower*, so this
notebook compares all three pipelines on the tiny model on this machine.

| Build | Model class | Pipeline | Main differences |
|---|---|---|---|
| CUDA (native) | `FastDLLMRelaxForCausalLM` | `opt_fast_dllm_llm_cublas` | cuBLAS + CUTLASS, PyTorch attention callback |
| Jetson Orin | same | same, `use_cutlass=False` | aarch64 host, sm_87, no CUTLASS (cross-compile), Matmul() rule dropped |
| Metal (Mac) | `FastDLLMRelaxForCausalLMNative` | `opt_fast_dllm_llm_native` | no vendor libraries, in-graph attention, static-shape siblings, int32 kernel indexing |

> **Metal is not ported to the in-place KV cache yet (plan 003).** The Native model still overrides
> the old functional-cache methods, so it no longer exports; the Metal cells below detect that and
> skip. Their committed outputs from before plan 003 describe the previous design.

**Needs:** TVM only for lowering. Building the Metal library needs TVM's Metal codegen; exporting the
Jetson library needs `aarch64-linux-gnu-g++`. Those cells skip with a message otherwise.

In [1]:
from _helpers import *
bootstrap()

repo root: fast-dllm-relax/  (cwd set)  tvm 0.24.dev0  torch 2.10.0+cu128


PosixPath('<repo>')

In [2]:
import collections
import tvm
from tvm import relax
from tvm.s_tir import dlight
from fast_dllm_relax.model.metal import (FastDLLMRelaxForCausalLMNative, MatmulSkipMixedPrecision,
                                          METAL_STATIC_FNS, metal_tir_pipeline)
from fast_dllm_relax.targets.metal import metal_target
from fast_dllm_relax.targets.jetson import jetson_target

cuda_target = tvm.target.Target.from_device(tvm.cuda(0))
_, mod_cuda, _ = export_model(tiny_config(tied=True), dtype="float16")
try:
    _, mod_metal, _ = export_model(tiny_config(tied=True), dtype="float16", cls=FastDLLMRelaxForCausalLMNative)
except Exception as err:  # noqa: BLE001 -- the Native model predates the in-place KV cache (plan 003)
    mod_metal = None
    print(f"Metal model does not export ({type(err).__name__}): not ported to the in-place KV cache yet.")
if mod_metal is not None:
    print("Metal-only functions:", sorted({g.name_hint for g in mod_metal.get_global_vars()} - {g.name_hint for g in mod_cuda.get_global_vars()}))
print("METAL_STATIC_FNS:", METAL_STATIC_FNS)

Metal-only functions: ['forward_cached_b32', 'forward_cached_block_b32', 'get_logits_m32', 'get_logits_m8']
METAL_STATIC_FNS: ('forward_cached_b32', 'forward_cached_block_b32', 'get_logits_m8', 'get_logits_m32')


## In-graph attention instead of the callback

Python callbacks into PyTorch cannot run on Metal, so the Native model overrides only the two cached
attention methods and computes attention inside the graph: concat, repeat GQA heads, fp32 QKᵀ,
softmax, fp32 PV, cast.

In [3]:
print("CUDA:"); show_ir(mod_cuda, "forward_cached", grep=r"attn_inplace", max_lines=4)
if mod_metal is not None:
    print("\nMetal:"); show_ir(mod_metal, "forward_cached", grep=r"R\.matmul|softmax|repeat", max_lines=8)

CUDA:
        attn_out2: R.Tensor((1, block_len, 4, 128), dtype="float16") = R.call_pure_packed("fast_dllm.flash_attn_bf16", add50, permute_dims56, permute_dims57, sinfo_args=(R.Tensor((1, block_len, 4, 128), dtype="float16"),))
    ...
        attn_out3: R.Tensor((1, block_len, 4, 128), dtype="float16") = R.call_pure_packed("fast_dllm.flash_attn_bf16", add57, permute_dims64, permute_dims65, sinfo_args=(R.Tensor((1, block_len, 4, 128), dtype="float16"),))

Metal:
        matmul37: R.Tensor((1, block_len, 1024), dtype="float16") = R.matmul(mul83, permute_dims55, out_dtype="void")
    ...
        repeat12: R.Tensor((1, 4, prefix_len + block_len, 128), dtype="float16") = R.repeat(concat22, repeats=2, axis=1)
        repeat13: R.Tensor((1, 4, prefix_len + block_len, 128), dtype="float16") = R.repeat(concat23, repeats=2, axis=1)
    ...
        permute_dims59: R.Tensor((1, 4, 128, prefix_len + block_len), dtype="float16") = R.permute_dims(repeat12, axes=[0, 1, 3, 2])
        matmul38: R.Ten

## Lowering all three

In [4]:
runs = {}
rec = make_recorder(keep_modules=True)
lower(mod_cuda, cuda_target, relax.get_pipeline("opt_fast_dllm_llm_cublas", use_cutlass=True, verbose=False), recorder=rec)
runs["CUDA"] = (rec, cuda_target, [dlight.gpu.Matmul(), dlight.gpu.GEMV(), dlight.gpu.Reduction(), dlight.gpu.GeneralReduction(), dlight.gpu.Fallback()])

rec = make_recorder(keep_modules=True)
jt = jetson_target()
lower(mod_cuda, jt, relax.get_pipeline("opt_fast_dllm_llm_cublas", use_cutlass=False, verbose=False), recorder=rec)
runs["Jetson"] = (rec, jt, [dlight.gpu.GEMV(), dlight.gpu.Reduction(), dlight.gpu.GeneralReduction(), dlight.gpu.Fallback()])

mt = metal_target()
lowered_metal = None
if mod_metal is not None:
    rec = make_recorder(keep_modules=True)
    lowered_metal = lower(mod_metal, mt, relax.get_pipeline("opt_fast_dllm_llm_native"), recorder=rec)
    runs["Metal"] = (rec, mt, [MatmulSkipMixedPrecision(), dlight.gpu.GEMV(), dlight.gpu.Reduction(), dlight.gpu.GeneralReduction(), dlight.gpu.Fallback()])

names = {k: [r.name for r in v[0].records if r.depth == 0] for k, v in runs.items()}
width = max(len(v) for v in names.values())
print_table([{t: (names[t][i] if i < len(names[t]) else "") for t in names} for i in range(width)])

[17:34:28] <repo>/3rdparty/tvm/include/tvm/topi/transform.h:1253: Warning: Fast mode segfaults when there are out-of-bounds indices. Make sure input indices are in bound
[17:34:32] <repo>/3rdparty/tvm/include/tvm/topi/transform.h:1253: Warning: Fast mode segfaults when there are out-of-bounds indices. Make sure input indices are in bound
[17:34:34] <repo>/3rdparty/tvm/include/tvm/topi/transform.h:1253: Warning: Fast mode segfaults when there are out-of-bounds indices. Make sure input indices are in bound


CUDA                    Jetson                  Metal                 
----------------------  ----------------------  ----------------------
FuseOpsByPattern        FuseOpsByPattern        FuseTransposeMatmul   
FuseOpsByPattern        RunCodegen              LegalizeOps           
RunCodegen              FuseTransposeMatmul     AnnotateTIROpPattern  
FuseTransposeMatmul     LegalizeOps             FoldConstant          
LegalizeOps             AnnotateTIROpPattern    FuseOps               
AnnotateTIROpPattern    FoldConstant            FuseTIR               
FoldConstant            FuseOps                 DeadCodeElimination   
FuseOps                 FuseTIR                 ApplyDefaultSchedule  
FuseTIR                 DeadCodeElimination     RewriteDataflowReshape
DeadCodeElimination     ApplyDefaultSchedule    ToNonDataflow         
ApplyDefaultSchedule    RewriteDataflowReshape  RemovePurityChecking  
RewriteDataflowReshape  ToNonDataflow           CallTIRRewrite        
ToNonD

In [5]:
rows = []
for label, (r, tgt, rules) in runs.items():
    last = r.records[-1].mod
    pick = dlight_rule_table(r.before("ApplyDefaultSchedule"), tgt, rules, show=False)
    rows.append({"build": label,
                 "cuBLAS fns": max((len(find_functions(x.mod, codegen="cublas")) for x in r.records if x.name == "FuseOpsByPattern"), default=0),
                 "CUTLASS fns": max((len(find_functions(x.mod, codegen="cutlass")) for x in r.records if x.name == "FuseOpsByPattern"), default=0),
                 "TIR kernels": ir_stats(last)["tir_fns"],
                 "dlight rules": dict(collections.Counter(pick.values()))})
print_table(rows)

build   cuBLAS fns  CUTLASS fns  TIR kernels  dlight rules                
------  ----------  -----------  -----------  ----------------------------
CUDA    7           1            77           {'(already scheduled)': 69} 
Jetson  7           0            77           {'(already scheduled)': 69} 
Metal   0           0            113          {'(already scheduled)': 102}


Without vendor libraries, every projection GEMM on Metal is a TIR kernel. They are scheduled by
`MatmulSkipMixedPrecision`: dlight's Matmul rule, except that mixed-precision (fp16 → fp32) attention
matmuls go to `Fallback`, because `Matmul()` hangs on them.

## Static-shape siblings and int32 indexing (Metal)

Relax shape variables are int64, so every symbolic-shape kernel indexes in 64-bit, which Metal
emulates slowly. The Metal build adds two things:
- fixed-row siblings (`forward_cached_b32`, `get_logits_m8`, ...), which compile to static-M GEMMs;
- an optional pass that narrows device-kernel indexing to int32. That pass caps the total sequence
  length (`NARROW_I32_MAX_TOTAL_LEN`).

In [6]:
from fast_dllm_relax.model.metal import NARROW_I32_MAX_TOTAL_LEN
if mod_metal is not None:
    for fn in ["forward_cached", "forward_cached_b32", "get_logits", "get_logits_m8"]:
        print(f"{fn:<26}", mod_metal[fn].params[0].struct_info)
print("int32 narrowing cap on total length:", NARROW_I32_MAX_TOTAL_LEN)

forward_cached_block       R.Tensor((1, block_len, 256), dtype="float16")
forward_cached_block_b32   R.Tensor((1, 32, 256), dtype="float16")
get_logits                 R.Tensor((1, seq_len, 256), dtype="float16")
get_logits_m8              R.Tensor((1, 8, 256), dtype="float16")
int32 narrowing cap on total length: 12288


## Building (optional, depends on the toolchain)

In [7]:
from pathlib import Path
from fast_dllm_relax.targets.metal import inspect_metal_tar
if lowered_metal is not None and env_report()["metal codegen"]:
    ex = build(lowered_metal, mt, tir_pipeline=metal_tir_pipeline(True))
    out = DUMP_DIR / "08_tiny_metal.tar"; out.parent.mkdir(parents=True, exist_ok=True)
    ex.export_library(str(out))
    info = inspect_metal_tar(out)
    print({k: v for k, v in info.items() if not isinstance(v, (list, dict))})
elif lowered_metal is None:
    print("Metal model not ported to the in-place KV cache (plan 003): skipping the Metal library build.")
else:
    print("TVM built without Metal codegen: lowering above still ran; skipping the Metal library build.")

[17:34:44] <repo>/3rdparty/tvm/src/target/opt/build_metal_off.cc:82: Warning: Metal runtime not enabled, return a source-only metal module (serializable for a Metal-enabled runtime)...


{'num_kernels': 112, 'long_count': 0, 'max_buffer_index': 4, 'max_threadgroup_bytes': 5120}


In [8]:
import shutil
from tvm.contrib import cc
from fast_dllm_relax.targets.jetson import JETSON_CROSS_CXX
lowered_jetson = runs["Jetson"][0].records[-1].mod
ex = build(lowered_jetson, jt)
print("Jetson tiny model compiled (sm_87 device code, aarch64 host code).")
if shutil.which(JETSON_CROSS_CXX):
    out = DUMP_DIR / "08_tiny_jetson.so"; out.parent.mkdir(parents=True, exist_ok=True)
    ex.export_library(str(out), fcompile=cc.cross_compiler(JETSON_CROSS_CXX))
    import subprocess
    print(subprocess.run(["file", "-b", str(out)], capture_output=True, text=True).stdout.strip())
else:
    print(f"{JETSON_CROSS_CXX} not found: skipping the cross-linked export.")

Jetson tiny model compiled (sm_87 device code, aarch64 host code).
ELF 64-bit LSB shared object, ARM aarch64, version 1 (SYSV), dynamically linked, BuildID[sha1]=3b0076808b44ff1631f84d5614b7612f0c874b17, with debug_info, not stripped


## Takeaways
- One model definition, three builds. The Jetson build changes only the target and the pattern set
  (no CUTLASS). The Metal build swaps in an attention implementation and a pipeline with no vendor
  libraries.
- Without cuBLAS, dlight schedules every GEMM, so static shapes and int32 indexing matter much more
  for speed.
- These lowerings can all be inspected on a Linux workstation. Only running the result needs the
  device.
- This is the last notebook. `README.md` in this folder has the index and conventions.